#Install Libraries

In [ ]:
!pip install tensorflow-datasets

#Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds

#Load MNIST (Auto Download)

In [ ]:
(X_train_mnist, y_train_mnist), (X_test_mnist, y_test_mnist) = tf.keras.datasets.mnist.load_data()

print("MNIST Train:", X_train_mnist.shape)
print("MNIST Test:", X_test_mnist.shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
MNIST Train: (60000, 28, 28)
MNIST Test: (10000, 28, 28)


#Load EMNIST (No manual download)

In [ ]:
ds_train, ds_test = tfds.load(
    'emnist/letters',
    split=['train', 'test'],
    as_supervised=True
)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/emnist/letters/incomplete.Z90D4R_3.1.0/emnist-train.tfrecord-[0-9][0-9][0-…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/emnist/letters/incomplete.Z90D4R_3.1.0/emnist-test.tfrecord-[0-9][0-9][0-9…

Dataset emnist downloaded and prepared to /root/tensorflow_datasets/emnist/letters/3.1.0. Subsequent calls will reuse this data.


#Convert EMNIST → NumPy (for CNN)

In [ ]:
def convert_tfds_to_numpy(dataset):
    images = []
    labels = []

    for img, label in tfds.as_numpy(dataset):
        images.append(img)
        labels.append(label)

    return np.array(images), np.array(labels)

X_train_emnist, y_train_emnist = convert_tfds_to_numpy(ds_train)
X_test_emnist, y_test_emnist = convert_tfds_to_numpy(ds_test)

print("EMNIST Train:", X_train_emnist.shape)
print("EMNIST Test:", X_test_emnist.shape)

EMNIST Train: (88800, 28, 28, 1)
EMNIST Test: (14800, 28, 28, 1)


#Normalize Data

In [ ]:
X_train_mnist = X_train_mnist / 255.0
X_test_mnist = X_test_mnist / 255.0

X_train_emnist = X_train_emnist / 255.0
X_test_emnist = X_test_emnist / 255.0

#Reshape for CNN

In [ ]:
X_train_mnist = X_train_mnist.reshape(-1, 28, 28, 1)
X_test_mnist = X_test_mnist.reshape(-1, 28, 28, 1)

X_train_emnist = X_train_emnist.reshape(-1, 28, 28, 1)
X_test_emnist = X_test_emnist.reshape(-1, 28, 28, 1)

#Fix EMNIST Labels

In [ ]:
y_train_emnist = y_train_emnist - 1
y_test_emnist = y_test_emnist - 1

#One-hot Encoding

In [ ]:
from tensorflow.keras.utils import to_categorical

y_train_mnist = to_categorical(y_train_mnist, 10)
y_test_mnist = to_categorical(y_test_mnist, 10)

y_train_emnist = to_categorical(y_train_emnist, 26)
y_test_emnist = to_categorical(y_test_emnist, 26)

#Build CNN Model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

def create_cnn(num_classes):

    model = Sequential()

    model.add(Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)))
    model.add(MaxPooling2D(2,2))

    model.add(Conv2D(64, (3,3), activation='relu'))
    model.add(MaxPooling2D(2,2))

    model.add(Flatten())

    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.3))

    model.add(Dense(num_classes, activation='softmax'))

    return model

#Train MNIST Model

In [ ]:
mnist_model = create_cnn(10)

mnist_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

mnist_model.fit(
    X_train_mnist,
    y_train_mnist,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 48s 60ms/step - accuracy: 0.6250 - loss: 1.0945 - val_accuracy: 0.9029 - val_loss: 0.3368
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 51s 68ms/step - accuracy: 0.8825 - loss: 0.3812 - val_accuracy: 0.9327 - val_loss: 0.2253
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 75s 59ms/step - accuracy: 0.9159 - loss: 0.2788 - val_accuracy: 0.9528 - val_loss: 0.1617
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 42s 56ms/step - accuracy: 0.9333 - loss: 0.2217 - val_accuracy: 0.9603 - val_loss: 0.1342
Epoch 5/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 82s 56ms/step - accuracy: 0.9443 - loss: 0.1851 - val_accuracy: 0.9658 - val_loss: 0.1197


# Train EMNIST Model

In [ ]:
emnist_model = create_cnn(26)

emnist_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

emnist_model.fit(
    X_train_emnist,
    y_train_emnist,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 69s 61ms/step - accuracy: 0.7611 - loss: 0.7782 - val_accuracy: 0.8889 - val_loss: 0.3361
Epoch 2/5
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 65s 58ms/step - accuracy: 0.8741 - loss: 0.3878 - val_accuracy: 0.9097 - val_loss: 0.2744
Epoch 3/5
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 81s 57ms/step - accuracy: 0.8960 - loss: 0.3182 - val_accuracy: 0.9188 - val_loss: 0.2457
Epoch 4/5
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 79s 55ms/step - accuracy: 0.9090 - loss: 0.2759 - val_accuracy: 0.9257 - val_loss: 0.2282
Epoch 5/5
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 62s 56ms/step - accuracy: 0.9168 - loss: 0.2478 - val_accuracy: 0.9296 - val_loss: 0.2149


#Evaluate Models

In [ ]:
mnist_loss, mnist_acc = mnist_model.evaluate(X_test_mnist, y_test_mnist)
print("MNIST Accuracy:", mnist_acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9647 - loss: 0.1151
MNIST Accuracy: 0.9646999835968018


In [ ]:
emnist_loss, emnist_acc = emnist_model.evaluate(X_test_emnist, y_test_emnist)
print("EMNIST Accuracy:", emnist_acc)

463/463 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9174 - loss: 0.2439
EMNIST Accuracy: 0.917432427406311


#save models

In [ ]:
mnist_model.save("mnist_model.keras")
emnist_model.save("emnist_model.keras")

#CRNN (CNN + LSTM + CTC)

In [ ]:
!pip install tensorflow

#CRNN Architecture

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D
from tensorflow.keras.layers import Reshape, Dense, LSTM, Bidirectional
from tensorflow.keras.models import Model

#Build CRNN Model

In [ ]:
def build_crnn():

    inputs = Input(shape=(28,28,1))

    # CNN Feature Extractor
    x = Conv2D(32, (3,3), activation='relu', padding='same')(inputs)
    x = MaxPooling2D(2,2)(x)

    x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = MaxPooling2D(2,2)(x)

    # Convert CNN output to sequence
    x = Reshape((7, 64*7))(x)

    # RNN for sequence learning
    x = Bidirectional(LSTM(128, return_sequences=True))(x)
    x = Bidirectional(LSTM(64, return_sequences=True))(x)

    # Output layer (sequence of characters)
    outputs = Dense(27, activation='softmax')(x)

    model = Model(inputs, outputs)

    return model